In [1]:
# A_make_degradation_features.py
import pandas as pd
import numpy as np
from sklearn.linear_model import HuberRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

# ===== 입력/출력 경로 =====
SRC = "maintenance_logs.csv"   # 시간순 로그 (장비별 시계열)
FEAT_OUT = "degradation_features.csv"
BASELINE_OUT = "expected_popup_baseline.joblib"

# ===== 하이퍼파라미터 =====
EWMA_ALPHA = 0.2        # HI 지수가중 (0.1~0.3 권장)
SLOPE_WINDOW = 48       # 최근 N창으로 기울기(예: 5분창이라면 4시간 = 48)
Z_EPS = 1e-9

# ===== 0) 데이터 로드 =====
df = pd.read_csv(SRC)
# 필요한 최소 컬럼 점검
need = ["DeviceID","Timestamp","WindSpeed_kn","Weather","Sensitivity","PopupCount"]
miss = [c for c in need if c not in df.columns]
if miss:
    raise ValueError(f"필수 컬럼 누락: {miss}")

# 타입/정렬
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values(["DeviceID","Timestamp"]).reset_index(drop=True)

# ===== 1) '환경→팝업수' 베이스라인(정상 기대치) 학습 =====
#  - 간단 회귀로 ExpectedPopup 예측 (환경/민감도 기반)
#  - 강건회귀 Huber + 원핫 인코딩(날씨)
X_cols_cat = ["Weather"]
X_cols_num = ["WindSpeed_kn","Sensitivity"]
y_col = "PopupCount"

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), X_cols_cat),
    ("num", "passthrough", X_cols_num)
])
reg = HuberRegressor(epsilon=1.5)
pipe = Pipeline([("pre", pre), ("reg", reg)])

# 베이스라인은 전체(또는 '정상기간'만)로 학습 – 초기에는 전체로 두고 운영에서 갱신 권장
pipe.fit(df[X_cols_cat + X_cols_num], df[y_col])
joblib.dump(pipe, BASELINE_OUT)

# ===== 2) 예측값/잔차/표준화 =====
df["ExpectedPopup"] = pipe.predict(df[X_cols_cat + X_cols_num])
df["Residual"] = df["PopupCount"] - df["ExpectedPopup"]

# 장비별 잔차 표준화(z-score) → 장비 고유 분산 차이 보정
def standardize_group(g):
    mu = g["Residual"].mean()
    sd = g["Residual"].std(ddof=0)
    sd = sd if sd > 0 else Z_EPS
    g["Residual_z"] = (g["Residual"] - mu) / sd
    return g

df = df.groupby("DeviceID", group_keys=False).apply(standardize_group)

# ===== 3) Health Index(EWMA) =====
def ewma(arr, alpha):
    out = np.zeros_like(arr, dtype=float)
    prev = 0.0
    for i, x in enumerate(arr):
        prev = alpha * x + (1 - alpha) * prev
        out[i] = prev
    return out

df["HI"] = 0.0
for did, g in df.groupby("DeviceID"):
    idx = g.index
    df.loc[idx, "HI"] = ewma(g["Residual_z"].values, EWMA_ALPHA)

# ===== 4) 최근 구간 기울기(드리프트 속도) =====
def rolling_slope(y, w):
    n = len(y)
    s = np.full(n, np.nan)
    x = np.arange(w)
    x_mean = x.mean()
    den = ((x - x_mean)**2).sum()
    for i in range(w-1, n):
        seg = y[i-w+1:i+1]
        y_mean = seg.mean()
        num = ((x - x_mean) * (seg - y_mean)).sum()
        s[i] = num / (den + 1e-9)
    return s

df["HI_Slope"] = 0.0
for did, g in df.groupby("DeviceID"):
    idx = g.index
    df.loc[idx, "HI_Slope"] = rolling_slope(g["HI"].values, SLOPE_WINDOW)

# ===== 5) 간단 경보 지표 (과민/둔감 방향성) =====
#  - HI > 0 이면 최근 과민 성향(예상보다 팝업↑), HI < 0 이면 둔감 성향
#  - Slope > 0 이면 과민 쪽으로 '진행', Slope < 0 이면 둔감 쪽으로 '진행'
df["DriftDirection"] = np.where(df["HI"] >= 0, "over", "under")
df["DriftSlopeSign"] = np.where(df["HI_Slope"] >= 0, "increasing", "decreasing")

# ===== 6) 출력 =====
keep = ["DeviceID","Timestamp","WindSpeed_kn","Weather","Sensitivity","PopupCount",
        "ExpectedPopup","Residual","Residual_z","HI","HI_Slope","DriftDirection","DriftSlopeSign"]
df[keep].to_csv(FEAT_OUT, index=False)
print("✅ saved:", FEAT_OUT)
print(df[keep].tail(8).to_string(index=False))


FileNotFoundError: [Errno 2] No such file or directory: 'maintenance_logs.csv'

In [ ]:
# B_predict_RUL.py
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import joblib

FEAT_IN = "degradation_features.csv"
BASELINE_PATH = "expected_popup_baseline.joblib"

# ===== 임계치 설정 (운영에서 튜닝) =====
HI_FAIL = 2.5          # 과민/둔감 임계 레벨(표준화 HI)
SLOPE_MIN = 0.002      # 유의미한 드리프트 속도 하한 (단위: HI/창)

# ===== 0) 로드 =====
df = pd.read_csv(FEAT_IN)
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

# ===== 1) 선형 외삽 기반 RUL(장비별 최신 시점) =====
def linear_rul_for_device(g, hi_fail=HI_FAIL, slope_min=SLOPE_MIN):
    g = g.sort_values("Timestamp")
    last = g.iloc[-1]
    hi = float(last["HI"])
    slope = float(last["HI_Slope"])
    # 드리프트가 거의 없으면 RUL 추정 불가(=무한대 취급)
    if np.isnan(slope) or abs(slope) < slope_min:
        return np.inf
    # 실패면: 과민(+) 방향인지, 둔감(-) 방향인지 임계치 부호 맞춰 계산
    target = hi_fail if slope > 0 else -hi_fail
    remain = target - hi
    if (slope > 0 and remain <= 0) or (slope < 0 and remain >= 0):
        return 0.0
    # 선형 외삽: 남은 HI / slope = 남은 창 수
    steps = remain / slope
    return float(steps)

rul_rows = []
for did, g in df.groupby("DeviceID"):
    rul_steps = linear_rul_for_device(g)
    rul_rows.append({"DeviceID": did, "RUL_steps_linear": rul_steps,
                     "last_ts": g["Timestamp"].max(),
                     "last_HI": g["HI"].iloc[-1],
                     "last_slope": g["HI_Slope"].iloc[-1]})
rul_df = pd.DataFrame(rul_rows)
print("=== Linear RUL (steps) ===")
print(rul_df.to_string(index=False))

# ===== 2) (선택) 지도학습 기반 RUL 회귀 =====
#  - 전제: 정비 이력으로 'DaysToMaintenance' 라벨이 df에 조인되어 있다고 가정
#    (없다면 이 블록은 건너뛰고 선형 외삽만 사용)
if "DaysToMaintenance" in df.columns:
    # 최신 윈도우만 사용(장비별 마지막 레코드)
    last_df = df.sort_values("Timestamp").groupby("DeviceID").tail(1).copy()

    # 특징: 최근 HI/HI_Slope, 최근 N창 통계(평균/표준편차 등)도 추가 가능
    FEATURES = ["HI","HI_Slope","Residual_z"]
    TARGET = "DaysToMaintenance"

    has_label = last_df.dropna(subset=FEATURES + [TARGET]).copy()
    if len(has_label) >= 20:  # 최소 표본
        X = has_label[FEATURES]
        y = has_label[TARGET].astype(float)

        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
        rfr = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
        rfr.fit(X_tr, y_tr)
        print("\nSupervised RUL MAE (test):", np.mean(np.abs(rfr.predict(X_te) - y_te)))
        joblib.dump(rfr, "rul_regressor.joblib")
        print("Saved rul_regressor.joblib")
    else:
        print("\n[WARN] 지도학습용 RUL 라벨 표본이 부족합니다. 선형 외삽을 사용하세요.")
else:
    print("\nNo 'DaysToMaintenance' label. Using linear extrapolation only.")
